In [1]:
import sys
import os
import importlib
from pathlib import Path
import polars as pl

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "Candidate_Package").is_dir():
    PROJECT_ROOT = PROJECT_ROOT / "Candidate_Package"
sys.path.append(str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src import silver
importlib.reload(silver)

print("Directorio de trabajo:", Path.cwd())

Directorio de trabajo: e:\Pruebas Tecnicas\Adidas\Manager Omnichannel\BusinessCase_ManagerOmniAnalytics 1\Candidate_Package


In [2]:
bronze_files = {f.stem: pl.read_parquet(f) for f in Path("data/bronze").glob("*.parquet")}
print("Fuentes cargadas desde bronze:", list(bronze_files.keys()))

Fuentes cargadas desde bronze: ['app_orders', 'currency_reference', 'ecomm_orders', 'franchise_orders', 'retail_orders', 'wholesale_orders']


In [3]:
fact_orders = silver.build_silver_layer(bronze_files)

print(fact_orders.shape)
fact_orders.head(10)

(1425316, 14)


channel,consumer_id,order_id,country,year_month,category,product_id,qty_ordered,qty_sold,unit_price,currency,revenue_local,order_status,adiclub
str,str,str,str,str,str,str,i64,i64,f64,str,f64,str,bool
"""app""","""192015""","""APP80600162480""","""AR""","""2025-12""","""Originals""","""1312""",2,2,127287.93,"""ARS""",254575.86,"""delivered""",true
"""app""","""439630""","""APP80600008593""","""BR""","""2025-04""","""Originals""","""1328""",3,3,635.53,"""BRL""",1906.59,"""delivered""",true
"""app""","""387471""","""APP80600051640""","""BR""","""2025-12""","""Outdoor""","""1137""",3,3,386.02,"""BRL""",1158.06,"""delivered""",true
"""app""","""537830""","""APP80600067520""","""BR""","""2025-09""","""Running""","""1008""",2,2,742.55,"""BRL""",1485.1,"""delivered""",true
"""app""","""834221""","""APP80600060423""","""MX""","""2025-01""","""Originals""","""1334""",5,5,1902.76,"""MXN""",9513.8,"""delivered""",true
"""app""","""493041""","""APP80600077500""","""AR""","""2025-04""","""Football""","""1218""",3,3,128345.39,"""ARS""",385036.17,"""delivered""",true
"""app""","""370843""","""APP80600034922""","""MX""","""2025-01""","""Originals""","""1320""",3,0,null,"""MXN""",0.0,"""processing""",true
"""app""","""249541""","""APP80600197736""","""CL""","""2025-09""","""Football""","""1205""",5,5,102457.68,"""CLP""",512288.4,"""delivered""",true
"""app""","""270255""","""APP80600004153""","""CL""","""2025-07""","""Originals""","""1302""",2,2,63550.73,"""CLP""",127101.46,"""returned""",true


In [4]:
# Validaciones clave
print("Países:", sorted(fact_orders["country"].unique().to_list()))
print("Categorías:", sorted(fact_orders["category"].unique().to_list()))
print("Estados:", sorted(fact_orders["order_status"].unique().to_list()))
print("Canales:", fact_orders["channel"].value_counts())
print("\nYear_month sample:", fact_orders["year_month"].unique().sort().to_list())
print("\nNulos por columna:\n", fact_orders.null_count())

Países: ['AR', 'BR', 'CL', 'CO', 'MX', 'PE']
Categorías: ['Football', 'Originals', 'Outdoor', 'Running']
Estados: ['cancelled', 'delivered', 'processing', 'returned']
Canales: shape: (5, 2)
┌───────────┬────────┐
│ channel   ┆ count  │
│ ---       ┆ ---    │
│ str       ┆ u32    │
╞═══════════╪════════╡
│ franchise ┆ 45450  │
│ wholesale ┆ 66866  │
│ ecomm     ┆ 404000 │
│ retail    ┆ 707000 │
│ app       ┆ 202000 │
└───────────┴────────┘

Year_month sample: ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12']

Nulos por columna:
 shape: (1, 14)
┌─────────┬─────────────┬──────────┬─────────┬───┬──────────┬──────────────┬─────────────┬─────────┐
│ channel ┆ consumer_id ┆ order_id ┆ country ┆ … ┆ currency ┆ revenue_loca ┆ order_statu ┆ adiclub │
│ ---     ┆ ---         ┆ ---      ┆ ---     ┆   ┆ ---      ┆ l            ┆ s           ┆ ---     │
│ u32     ┆ u32         ┆ u32      ┆ u32     ┆   ┆ u32      ┆ ---

In [7]:
wholesale_nulls = fact_orders.filter(
    (pl.col("channel") == "wholesale") & (pl.col("revenue_local").is_null())
)

print("Distribución por order_status:")
print(wholesale_nulls["order_status"].value_counts())

print("\n¿unit_price o qty_sold son los nulos?")
print(wholesale_nulls.select(["qty_sold", "unit_price", "qty_ordered"]).null_count())

print("\nMuestra de filas con revenue nulo:")
print(wholesale_nulls.select(["country", "category", "qty_ordered", "qty_sold", "unit_price", "order_status"]).head(10))

Distribución por order_status:
shape: (2, 2)
┌──────────────┬───────┐
│ order_status ┆ count │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ cancelled    ┆ 16732 │
│ processing   ┆ 16357 │
└──────────────┴───────┘

¿unit_price o qty_sold son los nulos?
shape: (1, 3)
┌──────────┬────────────┬─────────────┐
│ qty_sold ┆ unit_price ┆ qty_ordered │
│ ---      ┆ ---        ┆ ---         │
│ u32      ┆ u32        ┆ u32         │
╞══════════╪════════════╪═════════════╡
│ 0        ┆ 33089      ┆ 0           │
└──────────┴────────────┴─────────────┘

Muestra de filas con revenue nulo:
shape: (10, 6)
┌─────────┬───────────┬─────────────┬──────────┬────────────┬──────────────┐
│ country ┆ category  ┆ qty_ordered ┆ qty_sold ┆ unit_price ┆ order_status │
│ ---     ┆ ---       ┆ ---         ┆ ---      ┆ ---        ┆ ---          │
│ str     ┆ str       ┆ i64         ┆ i64      ┆ f64        ┆ str          │
╞═════════╪═══════════╪═════════════╪══════════╪════════════╪══

In [8]:
print("Categorías finales:", sorted(fact_orders["category"].unique().to_list()))
print("\nConteo por canal y categoría (debe verse balanceado, sin nulos):")
print(fact_orders.group_by(["channel", "category"]).agg(pl.len()).sort(["channel", "category"]))

Categorías finales: ['Football', 'Originals', 'Outdoor', 'Running']

Conteo por canal y categoría (debe verse balanceado, sin nulos):
shape: (20, 3)
┌───────────┬───────────┬────────┐
│ channel   ┆ category  ┆ len    │
│ ---       ┆ ---       ┆ ---    │
│ str       ┆ str       ┆ u32    │
╞═══════════╪═══════════╪════════╡
│ app       ┆ Football  ┆ 50692  │
│ app       ┆ Originals ┆ 50621  │
│ app       ┆ Outdoor   ┆ 50424  │
│ app       ┆ Running   ┆ 50263  │
│ ecomm     ┆ Football  ┆ 101117 │
│ …         ┆ …         ┆ …      │
│ retail    ┆ Running   ┆ 176900 │
│ wholesale ┆ Football  ┆ 16705  │
│ wholesale ┆ Originals ┆ 16760  │
│ wholesale ┆ Outdoor   ┆ 16749  │
│ wholesale ┆ Running   ┆ 16652  │
└───────────┴───────────┴────────┘


In [9]:
currency_ref = pl.read_parquet("data/bronze/currency_reference.parquet")
print(currency_ref.head(10))
print("\nPaíses:", sorted(currency_ref["country"].unique().to_list()))
print("Formato year_month:", sorted(currency_ref["year_month"].unique().to_list())[:3])
print("currency_from únicos:", currency_ref["currency_from"].unique().to_list())
print("currency_to únicos:", currency_ref["currency_to"].unique().to_list())

shape: (10, 7)
┌─────────┬────────────┬───────────────┬─────────────┬───────────────┬──────────────┬──────────────┐
│ country ┆ year_month ┆ currency_from ┆ currency_to ┆ local_units_p ┆ _source      ┆ _ingested_at │
│ ---     ┆ ---        ┆ ---           ┆ ---         ┆ er_eur        ┆ ---          ┆ ---          │
│ str     ┆ str        ┆ str           ┆ str         ┆ ---           ┆ str          ┆ str          │
│         ┆            ┆               ┆             ┆ f64           ┆              ┆              │
╞═════════╪════════════╪═══════════════╪═════════════╪═══════════════╪══════════════╪══════════════╡
│ AR      ┆ 2025-01    ┆ ARS           ┆ EUR         ┆ 1176.8699     ┆ currency_ref ┆ 2026-08-19T0 │
│         ┆            ┆               ┆             ┆               ┆ erence.csv   ┆ 6:22:37.0070 │
│         ┆            ┆               ┆             ┆               ┆              ┆ 71+00:…      │
│ AR      ┆ 2025-02    ┆ ARS           ┆ EUR         ┆ 1205.0466     ┆ curre

In [10]:
from src import currency

currency_ref = bronze_files["currency_reference"]
fact_orders_eur = currency.convert_to_eur(fact_orders, currency_ref)

print(fact_orders_eur.shape)
fact_orders_eur.select(["channel", "country", "currency", "revenue_local", "revenue_eur"]).head(10)

(1425316, 16)


channel,country,currency,revenue_local,revenue_eur
str,str,str,f64,f64
"""app""","""AR""","""ARS""",254575.86,223.020002
"""app""","""BR""","""BRL""",1906.59,299.368788
"""app""","""BR""","""BRL""",1158.06,180.87057
"""app""","""BR""","""BRL""",1485.1,252.22058
"""app""","""MX""","""MXN""",9513.8,492.549986
"""app""","""AR""","""ARS""",385036.17,317.280009
"""app""","""MX""","""MXN""",0.0,0.0
"""app""","""CL""","""CLP""",512288.4,505.100019
"""app""","""CL""","""CLP""",127101.46,131.799998


In [11]:
# Validación: revenue_eur debe ser razonablemente similar entre canales
# (todos en la misma unidad ahora), sin outliers absurdos
print(
    fact_orders_eur
    .group_by("channel")
    .agg([
        pl.col("revenue_eur").mean().alias("avg_revenue_eur"),
        pl.col("revenue_eur").null_count().alias("nulls"),
    ])
)

shape: (5, 3)
┌───────────┬─────────────────┬───────┐
│ channel   ┆ avg_revenue_eur ┆ nulls │
│ ---       ┆ ---             ┆ ---   │
│ str       ┆ f64             ┆ u32   │
╞═══════════╪═════════════════╪═══════╡
│ app       ┆ 244.688868      ┆ 0     │
│ ecomm     ┆ 212.679677      ┆ 0     │
│ franchise ┆ 263.432891      ┆ 0     │
│ retail    ┆ 250.171523      ┆ 0     │
│ wholesale ┆ 4278.786346     ┆ 33089 │
└───────────┴─────────────────┴───────┘


In [12]:
wholesale_check = fact_orders_eur.filter(pl.col("channel") == "wholesale")
print(wholesale_check.select(["qty_sold", "unit_price", "unit_price_eur", "revenue_eur"]).describe())

shape: (9, 5)
┌────────────┬───────────┬────────────┬────────────────┬─────────────┐
│ statistic  ┆ qty_sold  ┆ unit_price ┆ unit_price_eur ┆ revenue_eur │
│ ---        ┆ ---       ┆ ---        ┆ ---            ┆ ---         │
│ str        ┆ f64       ┆ f64        ┆ f64            ┆ f64         │
╞════════════╪═══════════╪════════════╪════════════════╪═════════════╡
│ count      ┆ 66866.0   ┆ 33777.0    ┆ 33777.0        ┆ 33777.0     │
│ null_count ┆ 0.0       ┆ 33089.0    ┆ 33089.0        ┆ 33089.0     │
│ mean       ┆ 18.389391 ┆ 117.526335 ┆ 117.526335     ┆ 4278.786346 │
│ std        ┆ 28.602963 ┆ 21.995191  ┆ 21.995191      ┆ 3710.88903  │
│ min        ┆ 0.0       ┆ 44.33      ┆ 44.33          ┆ 45.38       │
│ 25%        ┆ 0.0       ┆ 105.06     ┆ 105.06         ┆ 1026.3      │
│ 50%        ┆ 1.0       ┆ 114.43     ┆ 114.43         ┆ 3391.36     │
│ 75%        ┆ 30.0      ┆ 129.09     ┆ 129.09         ┆ 6793.44     │
│ max        ┆ 182.0     ┆ 219.33     ┆ 219.33         ┆ 23265.